# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list the available Record Sets and their associated Fields and Columns by unique `@id` references.

> **Note:** All entities are referenced by their Croissant `@id`. This helps to ensure unambiguous identification throughout processing.

Let's inspect the record sets present in this dataset:

In [ ]:
# List all available record sets by @id and title
record_sets = dataset.metadata.record_sets  # this is a list of RecordSet objects
print(f"There are {len(record_sets)} record sets in this dataset.")

for rs in record_sets:
    print(f'-- RecordSet @id: {rs.id}')
    print(f'   name: {getattr(rs, "name", None)}')
    # List the fields and columns for this record set
    if hasattr(rs, 'fields') and rs.fields:
        print("   Fields:")
        for f in rs.fields:
            print(f'      Field @id: {f.id}\tname: {getattr(f, "name", None)}\ttype: {getattr(f, "data_type", None)}')
    if hasattr(rs, "columns") and rs.columns:
        print("   Columns:")
        for col in rs.columns:
            print(f'      Column @id: {col.id}\tname: {getattr(col,"name",None)}\ttype: {getattr(col, "data_type", None)}')
    print()

You can see the available `@id`s and names for each record set and their fields/columns above.

## View sample records from each RecordSet

Let's print a few sample records from the first available record set (`@id`).

In [ ]:
# Inspect the first record set's ID
if len(record_sets) == 0:
    raise Exception("No record sets found in this dataset.")

# Pick first record set
sample_record_set = record_sets[0].id
print(f'Showing example records for record set: {sample_record_set}')

# Print several records from this record set
for i, record in enumerate(dataset.records(record_set=sample_record_set)):
    print(f'Record {i}: {record}')
    if i > 2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use Record Set `@id`s as identifiers.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

# We'll load each record set to a pandas DataFrame
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"  No records for {record_set_id}")
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
    print(f"  Example data (first 3 rows):")
    print(dataframes[record_set_id].head(3))

Choose a record set and field for further EDA (you may need to update the IDs according to the output above).

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from the main protein abundance record set for filtering, normalization, and grouping.

> **Note:** Replace the record set and field `@id` below if you want to analyze a different part of the dataset.

In [ ]:
# Choose main protein record set, e.g.:
main_record_set = record_set_ids[0]  # update this index as desired
df = dataframes[main_record_set]
print(f"Working with record set: {main_record_set}")
print(df.head(3))

# List all numeric columns to help user pick
print("Numeric columns:")
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(numeric_fields)

# Choose a numeric field for filtering (e.g., 'coverage' or whatever matches the data)
if numeric_fields:
    numeric_field = numeric_fields[0]  # e.g., 'coverage' or 'PeptideCount', update as necessary
else:
    # If there are no auto-recognized numeric fields, pick a common-sensical one
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field for EDA: {numeric_field}")
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
if threshold is None:
    raise Exception(f"Column {numeric_field} is not numeric for filtering.")

# Filter records where value > mean
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field if present (e.g., 'Sample', 'Replicate', etc.)
categorical_fields = [col for col in df.columns if pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype == object]
group_field = None
for candidate in ['Sample', 'Condition', 'accession', 'description']:
    if candidate in categorical_fields:
        group_field = candidate
        break
if not group_field and categorical_fields:
    group_field = categorical_fields[0]
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical grouping field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we'll use matplotlib and seaborn to plot the normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
plt.title(f'Histogram of normalized {numeric_field} values')
plt.xlabel(f'Normalized {numeric_field}')
plt.ylabel('Frequency')
plt.show()

# Optional: plot group means if grouped
if group_field:
    plt.figure(figsize=(12,6))
    grouped_df.plot(kind='bar')
    plt.title(f'Mean {numeric_field} per group ({group_field})')
    plt.ylabel(f'Mean {numeric_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR<sup>2</sup> dataset *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* using `mlcroissant`. We inspected available record sets and fields by their `@id`, extracted records into pandas DataFrames, and demonstrated filtering, normalization, grouping, and visualization over quantitative fields.

**Next steps:**
- Perform deeper statistical analysis or machine learning on the data.
- Explore domain-specific relationships or patterns depending on your research questions.
- Review the Croissant schema for additional metadata and annotation.

For more, see [`mlcroissant` documentation](https://mlcommons.org/croissant/) and the [dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa/).